# Ingestion Pipeline Demo

This notebook demonstrates Valorem's Ingestion Pipeline (Phase 4, Milestones 11-12):

1. **Options Manifest Generator** - Deterministic symbol selection with DTE/moneyness filters
2. **Ingestion Orchestrator** - Multi-source data ingestion with cost estimation and validation

In [47]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
import json

# Ingestion Pipeline components
from src.data.ingest.manifest import (
    ManifestGenerator, 
    OptionSymbolInfo,
    Manifest,
    get_spot_reference,
    compute_dte,
    compute_moneyness,
)
from src.data.ingest.orchestrator import (
    IngestionOrchestrator,
    IngestionResult,
    CostExceededException,
    DataQualityException,
)

pd.set_option('display.float_format', '{:.4f}'.format)
print("Ingestion Pipeline components loaded successfully!")

Ingestion Pipeline components loaded successfully!


## 1. Options Symbol Parsing (OSI Format)

The `ManifestGenerator` uses the OSI (Options Symbology Initiative) format to parse option symbols:

```
{underlying}{YYMMDD}{C/P}{strike*1000}
```

Example: `SPY250117C00500000` = SPY Jan 17, 2025 $500 Call

In [48]:
from src.config.schema import ConfigSchema, DatasetConfig, DatasetSplitsConfig, BacktestConfig
from datetime import date

# Create config with required fields for demonstration
# (In production, this would be loaded from config.yaml)
config = ConfigSchema(
    dataset=DatasetConfig(
        splits=DatasetSplitsConfig(
            train_start=date(2023, 1, 1),
            train_end=date(2023, 12, 31),
            val_start=date(2024, 1, 1),
            val_end=date(2024, 3, 31),
            test_start=date(2024, 4, 1),
            test_end=date(2024, 6, 30),
        ),
        min_dte=7,
        max_dte=120,
    ),
    backtest=BacktestConfig(
        start_date=date(2024, 1, 1),
        end_date=date(2024, 6, 30),
    ),
)

# Initialize manifest generator
generator = ManifestGenerator(config=config)

# Example OSI symbols
sample_symbols = [
    "SPY250117C00500000",  # SPY Jan 17 2025 $500 Call
    "SPY250117P00450000",  # SPY Jan 17 2025 $450 Put
    "SPY250221C00480000",  # SPY Feb 21 2025 $480 Call
    "SPY250221P00460000",  # SPY Feb 21 2025 $460 Put
    "SPY250321C00510000",  # SPY Mar 21 2025 $510 Call
    "INVALID_SYMBOL",      # Invalid format
]

print("Parsing OSI option symbols:\n")
print(f"{'Symbol':<22} {'Underlying':>10} {'Expiry':>12} {'Right':>6} {'Strike':>10}")
print("="*65)

for symbol in sample_symbols:
    info = generator.parse_option_symbol(symbol)
    if info:
        print(f"{symbol:<22} {info.underlying:>10} {info.expiry_date.isoformat():>12} {info.right:>6} ${info.strike:>8.2f}")
    else:
        print(f"{symbol:<22} {'FAILED TO PARSE':>46}")

Parsing OSI option symbols:

Symbol                 Underlying       Expiry  Right     Strike
SPY250117C00500000            SPY   2025-01-17      C $  500.00
SPY250117P00450000            SPY   2025-01-17      P $  450.00
SPY250221C00480000            SPY   2025-02-21      C $  480.00
SPY250221P00460000            SPY   2025-02-21      P $  460.00
SPY250321C00510000            SPY   2025-03-21      C $  510.00
INVALID_SYMBOL                                        FAILED TO PARSE


## 2. DTE and Moneyness Calculations

Key filtering criteria for manifest generation:
- **DTE (Days to Expiration)**: Typically 7-120 days
- **Moneyness (Strike/Spot)**: Typically 0.8-1.2 (20% OTM puts to 20% OTM calls)

In [49]:
# Current reference point
as_of_date = date(2025, 1, 6)
spot_price = 480.0

# Parse symbols and compute metrics
print(f"Reference: as_of={as_of_date}, spot=${spot_price:.2f}\n")
print(f"{'Symbol':<22} {'Expiry':<12} {'Strike':>8} {'DTE':>5} {'Moneyness':>10}")
print("="*60)

for symbol in sample_symbols[:-1]:  # Skip invalid
    info = generator.parse_option_symbol(symbol)
    if info:
        dte = compute_dte(info.expiry_date, as_of_date)
        moneyness = compute_moneyness(info.strike, spot_price)
        print(f"{symbol:<22} {info.expiry_date.isoformat():<12} ${info.strike:>6.0f} {dte:>5}d {moneyness:>10.3f}")

Reference: as_of=2025-01-06, spot=$480.00

Symbol                 Expiry         Strike   DTE  Moneyness
SPY250117C00500000     2025-01-17   $   500    11d      1.042
SPY250117P00450000     2025-01-17   $   450    11d      0.938
SPY250221C00480000     2025-02-21   $   480    46d      1.000
SPY250221P00460000     2025-02-21   $   460    46d      0.958
SPY250321C00510000     2025-03-21   $   510    74d      1.062


In [50]:
# Demonstrate filtering
print("Filtering examples:\n")

# Filter criteria
dte_min, dte_max = 7, 90
moneyness_min, moneyness_max = 0.90, 1.10

print(f"Criteria: DTE in [{dte_min}, {dte_max}], Moneyness in [{moneyness_min:.2f}, {moneyness_max:.2f}]\n")

for symbol in sample_symbols[:-1]:
    info = generator.parse_option_symbol(symbol)
    if info:
        dte = compute_dte(info.expiry_date, as_of_date)
        moneyness = compute_moneyness(info.strike, spot_price)
        
        dte_pass = dte_min <= dte <= dte_max
        moneyness_pass = moneyness_min <= moneyness <= moneyness_max
        
        status = "PASS" if (dte_pass and moneyness_pass) else "FAIL"
        reasons = []
        if not dte_pass:
            reasons.append(f"DTE={dte}")
        if not moneyness_pass:
            reasons.append(f"M={moneyness:.3f}")
        
        reason_str = f" ({', '.join(reasons)})" if reasons else ""
        print(f"{symbol}: {status}{reason_str}")

Filtering examples:

Criteria: DTE in [7, 90], Moneyness in [0.90, 1.10]

SPY250117C00500000: PASS
SPY250117P00450000: PASS
SPY250221C00480000: PASS
SPY250221P00460000: PASS
SPY250321C00510000: PASS


## 3. Full Manifest Generation

The `ManifestGenerator` combines all filters and applies per-expiry-side caps to select the most relevant options (nearest to ATM first).

In [51]:
# Generate a realistic set of option symbols
def generate_sample_symbols(underlying: str, spot: float, as_of: date, num_expiries: int = 6):
    """Generate sample OSI symbols for demonstration."""
    symbols = []
    
    # Generate expiries (weeklies and monthlies)
    expiries = []
    for i in range(1, num_expiries + 1):
        # Weekly expiries (7, 14, 21 days)
        if i <= 3:
            expiry = as_of + timedelta(days=7*i)
        else:
            # Monthly expiries (30, 60, 90 days)
            expiry = as_of + timedelta(days=30*(i-2))
        expiries.append(expiry)
    
    # Generate strikes around spot
    strike_step = 5.0
    strikes = np.arange(spot * 0.75, spot * 1.25, strike_step)
    
    for expiry in expiries:
        for strike in strikes:
            for right in ['C', 'P']:
                # Format: {underlying}{YYMMDD}{C/P}{strike*1000:08d}
                symbol = f"{underlying}{expiry.strftime('%y%m%d')}{right}{int(strike*1000):08d}"
                symbols.append(symbol)
    
    return symbols

# Generate symbols
as_of_date = date(2025, 1, 6)
spot = 480.0
available_symbols = generate_sample_symbols("SPY", spot, as_of_date)

print(f"Generated {len(available_symbols)} sample option symbols")
print(f"\nSample symbols:")
for s in available_symbols[:5]:
    print(f"  {s}")
print(f"  ...")

Generated 576 sample option symbols

Sample symbols:
  SPY250113C00360000
  SPY250113P00360000
  SPY250113C00365000
  SPY250113P00365000
  SPY250113C00370000
  ...


In [52]:
# Generate manifest with filters
manifest = generator.generate_manifest(
    available_symbols=available_symbols,
    spot_reference=spot,
    as_of_date=as_of_date,
    dte_min=7,
    dte_max=90,
    moneyness_min=0.90,
    moneyness_max=1.10,
    options_per_expiry_side=10,  # Top 10 nearest-ATM per expiry per side
)

print("Manifest Generation Results")
print("="*50)
print(f"Input symbols:        {len(available_symbols)}")
print(f"Selected symbols:     {manifest.metadata.symbols_count}")
print(f"Expiries covered:     {manifest.metadata.expiries_count}")
print(f"Spot reference:       ${manifest.metadata.spot_reference:.2f}")
print(f"DTE range:            [{manifest.metadata.dte_min}, {manifest.metadata.dte_max}]")
print(f"Moneyness range:      [{manifest.metadata.moneyness_min:.2f}, {manifest.metadata.moneyness_max:.2f}]")
print(f"Per-expiry-side cap:  {manifest.metadata.options_per_expiry_side}")
print(f"Config hash:          {manifest.metadata.config_hash}")

Manifest Generation Results
Input symbols:        576
Selected symbols:     100
Expiries covered:     5
Spot reference:       $480.00
DTE range:            [7, 90]
Moneyness range:      [0.90, 1.10]
Per-expiry-side cap:  10
Config hash:          5d471db94e0175ff


In [53]:
# Show distribution by expiry
print("\nSymbols by Expiry:")
print(f"{'Expiry':<12} {'Calls':>6} {'Puts':>6} {'Total':>7}")
print("-"*35)

for expiry_str, symbols in manifest.symbols_by_expiry.items():
    calls = sum(1 for s in symbols if 'C' in s[9:10])
    puts = len(symbols) - calls
    print(f"{expiry_str:<12} {calls:>6} {puts:>6} {len(symbols):>7}")

total_calls = sum(1 for s in manifest.symbols if generator.parse_option_symbol(s).right == 'C')
total_puts = len(manifest.symbols) - total_calls
print("-"*35)
print(f"{'TOTAL':<12} {total_calls:>6} {total_puts:>6} {len(manifest.symbols):>7}")


Symbols by Expiry:
Expiry        Calls   Puts   Total
-----------------------------------
2025-01-13       10     10      20
2025-01-20       10     10      20
2025-01-27       10     10      20
2025-03-07       10     10      20
2025-04-06       10     10      20
-----------------------------------
TOTAL            50     50     100


## 4. Manifest Serialization

Manifests are saved as JSON for reproducibility and auditing.

In [54]:
# Convert to dictionary for display
manifest_dict = manifest.to_dict()

print("Manifest JSON Structure:")
print("="*50)

# Show metadata
print("\nmetadata:")
for key, value in manifest_dict['metadata'].items():
    print(f"  {key}: {value}")

# Show sample symbols
print(f"\nsymbols: [{len(manifest_dict['symbols'])} items]")
for s in manifest_dict['symbols'][:5]:
    print(f"  {s}")
print("  ...")

Manifest JSON Structure:

metadata:
  as_of_ts_utc: 2025-01-06T00:00:00
  spot_reference: 480.0
  dte_min: 7
  dte_max: 90
  moneyness_min: 0.9
  moneyness_max: 1.1
  options_per_expiry_side: 10
  underlying: SPY
  generated_at: 2026-01-26T03:13:52.198044
  config_hash: 5d471db94e0175ff
  symbols_count: 100
  expiries_count: 5

symbols: [100 items]
  SPY250113C00455000
  SPY250113C00460000
  SPY250113C00465000
  SPY250113C00470000
  SPY250113C00475000
  ...


In [55]:
# Demonstrate determinism - same inputs produce same output
manifest2 = generator.generate_manifest(
    available_symbols=available_symbols,
    spot_reference=spot,
    as_of_date=as_of_date,
    dte_min=7,
    dte_max=90,
    moneyness_min=0.90,
    moneyness_max=1.10,
    options_per_expiry_side=10,
)

# Compare symbols
symbols_match = manifest.symbols == manifest2.symbols
hash_match = manifest.metadata.config_hash == manifest2.metadata.config_hash

print("Determinism Verification:")
print(f"  Symbols match:     {symbols_match}")
print(f"  Config hash match: {hash_match}")
print(f"\nManifest generation is {'DETERMINISTIC' if symbols_match else 'NOT deterministic'}!")

Determinism Verification:
  Symbols match:     True
  Config hash match: True

Manifest generation is DETERMINISTIC!


## 5. Spot Reference Lookup

The `get_spot_reference` function performs a time-aware lookup to find the last close price at or before the reference timestamp.

In [56]:
# Create sample underlying data
underlying_data = pd.DataFrame({
    'ts_utc': pd.date_range('2025-01-01', '2025-01-10', freq='D'),
    'open': [475, 476, 478, 479, 480, 481, 482, 483, 484, 485],
    'high': [478, 479, 481, 482, 483, 484, 485, 486, 487, 488],
    'low':  [473, 474, 476, 477, 478, 479, 480, 481, 482, 483],
    'close': [476, 478, 479, 480, 481, 482, 483, 484, 485, 486],
    'volume': [1e6]*10,
})

print("Underlying Price Data:")
print(underlying_data[['ts_utc', 'close']].to_string(index=False))

# Test spot reference lookup
print("\nSpot Reference Lookups:")
for test_date in ['2025-01-05', '2025-01-06', '2025-01-10']:
    as_of_ts = datetime.fromisoformat(test_date + 'T16:00:00')
    spot = get_spot_reference(underlying_data, as_of_ts)
    print(f"  as_of={test_date}: spot=${spot:.2f}")

Underlying Price Data:
    ts_utc  close
2025-01-01    476
2025-01-02    478
2025-01-03    479
2025-01-04    480
2025-01-05    481
2025-01-06    482
2025-01-07    483
2025-01-08    484
2025-01-09    485
2025-01-10    486

Spot Reference Lookups:
  as_of=2025-01-05: spot=$481.00
  as_of=2025-01-06: spot=$482.00
  as_of=2025-01-10: spot=$486.00


## 6. Ingestion Orchestrator Overview

The `IngestionOrchestrator` coordinates the full data ingestion workflow:

1. **Generate/Load Manifest** - Select option symbols
2. **Estimate Costs** - Check against budget limits
3. **Fetch Underlying** - OHLCV bars from Databento
4. **Fetch Options** - Quote data from Databento  
5. **Fetch Macro** - FRED series (VIX, rates, etc.)
6. **Validate Data** - Quality checks
7. **Write to Storage** - Persist to DuckDB
8. **Log Metadata** - Audit trail

In [57]:
# Illustrate the workflow (using mock components)
print("Ingestion Workflow Steps:")
print("="*60)

steps = [
    ("1. Generate run_id", "Unique UUID for tracking"),
    ("2. Load/generate manifest", "Select option symbols via DTE/moneyness filters"),
    ("3. Estimate costs", "Query Databento API for data size estimates"),
    ("4. Check cost limit", "Reject if estimate > max_usd budget"),
    ("5. Fetch underlying bars", "OHLCV-1m from Databento (equities dataset)"),
    ("6. Fetch option quotes", "NBBO quotes from Databento (options dataset)"),
    ("7. Fetch macro series", "VIX, Treasury rates from FRED"),
    ("8. Validate data quality", "Check for nulls, duplicates, OHLC consistency"),
    ("9. Write to storage", "DuckDB tables: underlying_bars, option_quotes, fred_series"),
    ("10. Log ingestion run", "Metadata: run_id, cost, row counts, git SHA"),
]

for step, description in steps:
    print(f"{step:<30} {description}")

Ingestion Workflow Steps:
1. Generate run_id             Unique UUID for tracking
2. Load/generate manifest      Select option symbols via DTE/moneyness filters
3. Estimate costs              Query Databento API for data size estimates
4. Check cost limit            Reject if estimate > max_usd budget
5. Fetch underlying bars       OHLCV-1m from Databento (equities dataset)
6. Fetch option quotes         NBBO quotes from Databento (options dataset)
7. Fetch macro series          VIX, Treasury rates from FRED
8. Validate data quality       Check for nulls, duplicates, OHLC consistency
9. Write to storage            DuckDB tables: underlying_bars, option_quotes, fred_series
10. Log ingestion run          Metadata: run_id, cost, row counts, git SHA


In [58]:
# IngestionResult structure
print("\nIngestionResult Fields:")
print("="*50)

# Create a sample result
sample_result = IngestionResult(
    run_id="abc12345-6789-0123-4567-890abcdef123",
    start_time=datetime(2025, 1, 6, 10, 0, 0),
    end_time=datetime(2025, 1, 6, 10, 5, 32),
    preview_only=False,
    estimated_cost=12.50,
    actual_cost=12.50,
    underlying_rows=5000,
    option_rows=150000,
    macro_rows=50,
    validation_results={},
    errors=[],
)

print(f"run_id:          {sample_result.run_id[:8]}...")
print(f"duration:        {(sample_result.end_time - sample_result.start_time).seconds} seconds")
print(f"preview_only:    {sample_result.preview_only}")
print(f"estimated_cost:  ${sample_result.estimated_cost:.2f}")
print(f"actual_cost:     ${sample_result.actual_cost:.2f}")
print(f"underlying_rows: {sample_result.underlying_rows:,}")
print(f"option_rows:     {sample_result.option_rows:,}")
print(f"macro_rows:      {sample_result.macro_rows:,}")
print(f"total_rows:      {sample_result.total_rows:,}")
print(f"success:         {sample_result.success}")
print(f"\nSummary: {sample_result.summary()}")


IngestionResult Fields:
run_id:          abc12345...
duration:        332 seconds
preview_only:    False
estimated_cost:  $12.50
actual_cost:     $12.50
underlying_rows: 5,000
option_rows:     150,000
macro_rows:      50
total_rows:      155,050
success:         True

Summary: Ingestion SUCCESS (run_id=abc12345): 155050 total rows (underlying=5000, options=150000, macro=50), cost=$12.50


## 7. Data Quality Validation

The `DataQualityValidator` performs checks on ingested data before writing to storage.

In [59]:
from src.data.quality.validators import (
    DataQualityValidator,
    ValidationResult,
    ValidationIssue,
    IssueSeverity,
)

validator = DataQualityValidator()

print("Data Quality Checks:")
print("="*50)

checks = [
    ("Underlying Bars", [
        "Non-null required columns (ts_utc, open, high, low, close)",
        "Timestamps in ascending order",
        "No duplicates on (ts_utc, symbol)",
        "OHLC relationships (high >= low, etc.)",
        "Positive volume",
    ]),
    ("Option Quotes", [
        "Non-null required columns (ts_utc, option_symbol, bid, ask)",
        "No crossed quotes (bid > ask)",
        "Non-negative bid/ask prices",
        "Valid spreads (ask >= bid)",
    ]),
    ("FRED Series", [
        "Non-null date and value columns",
        "Dates in ascending order",
        "No duplicate dates per series",
    ]),
]

for category, rules in checks:
    print(f"\n{category}:")
    for rule in rules:
        print(f"  - {rule}")

Data Quality Checks:

Underlying Bars:
  - Non-null required columns (ts_utc, open, high, low, close)
  - Timestamps in ascending order
  - No duplicates on (ts_utc, symbol)
  - OHLC relationships (high >= low, etc.)
  - Positive volume

Option Quotes:
  - Non-null required columns (ts_utc, option_symbol, bid, ask)
  - No crossed quotes (bid > ask)
  - Non-negative bid/ask prices
  - Valid spreads (ask >= bid)

FRED Series:
  - Non-null date and value columns
  - Dates in ascending order
  - No duplicate dates per series


In [60]:
# Create sample data with some quality issues
good_underlying = pd.DataFrame({
    'ts_utc': pd.date_range('2025-01-06 09:30', periods=10, freq='1min'),
    'symbol': 'SPY',
    'open': [480, 481, 482, 483, 484, 485, 486, 487, 488, 489],
    'high': [481, 482, 483, 484, 485, 486, 487, 488, 489, 490],
    'low':  [479, 480, 481, 482, 483, 484, 485, 486, 487, 488],
    'close': [480.5, 481.5, 482.5, 483.5, 484.5, 485.5, 486.5, 487.5, 488.5, 489.5],
    'volume': [1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900],
})

# Run validation
result = validator.check_underlying_bars(good_underlying)

print("Validation Result (Good Data):")
print(f"  passed: {result.passed}")
print(f"  total_rows: {result.total_rows}")
print(f"  issues: {len(result.issues)}")

Validation Result (Good Data):
  passed: True
  total_rows: 10
  issues: 0


In [61]:
# Now with bad data
bad_underlying = pd.DataFrame({
    'ts_utc': pd.date_range('2025-01-06 09:30', periods=5, freq='1min'),
    'symbol': 'SPY',
    'open': [480, None, 482, 483, 484],    # Null value
    'high': [479, 482, 483, 484, 485],     # High < Open (first row)
    'low':  [481, 480, 481, 482, 483],     # Low > Open (first row)
    'close': [480.5, 481.5, 482.5, 483.5, 484.5],
    'volume': [1000, 1100, -100, 1300, 1400],  # Negative volume
})

result = validator.check_underlying_bars(bad_underlying)

print("Validation Result (Bad Data):")
print(f"  passed: {result.passed}")
print(f"  total_rows: {result.total_rows}")
print(f"  issues: {len(result.issues)}")

if result.issues:
    print("\n  Issue Details:")
    for issue in result.issues:
        print(f"    [{issue.severity.name}] {issue.check_name}: {issue.message}")

Validation Result (Bad Data):
  passed: False
  total_rows: 5
  issues: 2

  Issue Details:
    [ERROR] null_values: Column 'open' has 1 null values (20.00%)
    [ERROR] ohlc_relationships: Invalid OHLC relationships in 1 rows


## 8. CLI Script Usage

The ingestion pipeline can be run via CLI script:

```bash
# Preview mode (estimate costs, no data fetch)
python scripts/ingest_raw.py --preview-only --start-date 2024-01-01 --end-date 2024-01-02

# Full ingestion
python scripts/ingest_raw.py --start-date 2024-01-01 --end-date 2024-01-02

# With mock providers (no API keys needed)
python scripts/ingest_raw.py --mock --start-date 2024-01-01 --end-date 2024-01-02

# Skip data validation
python scripts/ingest_raw.py --skip-validation --start-date 2024-01-01 --end-date 2024-01-02
```

In [62]:
# Show CLI help (if script exists)
import subprocess
from pathlib import Path

script_path = Path('../scripts/ingest_raw.py')
if script_path.exists():
    print("CLI Script Available:")
    print(f"  Location: scripts/ingest_raw.py")
    print(f"\nRun 'python scripts/ingest_raw.py --help' for full options.")
else:
    print("CLI script not yet created.")
    print("To create, implement scripts/ingest_raw.py with argparse and orchestrator calls.")

CLI Script Available:
  Location: scripts/ingest_raw.py

Run 'python scripts/ingest_raw.py --help' for full options.


## 9. Summary

The Ingestion Pipeline (Phase 4) provides:

1. **ManifestGenerator** (M11)
   - OSI symbol parsing
   - DTE and moneyness filtering
   - Per-expiry-side caps (nearest-ATM selection)
   - Deterministic output (config hash)
   - JSON serialization for reproducibility

2. **IngestionOrchestrator** (M12)
   - Cost estimation and budget enforcement
   - Preview mode for dry runs
   - Multi-source data fetching (Databento, FRED)
   - Data quality validation
   - Storage persistence (DuckDB)
   - Audit logging (run metadata, git SHA)

In [63]:
print("="*60)
print("INGESTION PIPELINE DEMO COMPLETE")
print("="*60)
print("\nKey components demonstrated:")
print("  - ManifestGenerator: Symbol selection with DTE/moneyness filters")
print("  - OSI symbol parsing: SPY250117C00500000 format")
print("  - Deterministic manifest generation")
print("  - DataQualityValidator: Pre-write data checks")
print("  - IngestionOrchestrator workflow: 10-step pipeline")
print("\nNext: Use notebooks/03_feature_engineering_demo.ipynb for Phase 5")

INGESTION PIPELINE DEMO COMPLETE

Key components demonstrated:
  - ManifestGenerator: Symbol selection with DTE/moneyness filters
  - OSI symbol parsing: SPY250117C00500000 format
  - Deterministic manifest generation
  - DataQualityValidator: Pre-write data checks
  - IngestionOrchestrator workflow: 10-step pipeline

Next: Use notebooks/03_feature_engineering_demo.ipynb for Phase 5
